# Xumo Manual RAG Reflection Assistant

## What It Is 
This notebook implements a **Reflection-based Retrieval-Augmented Generation (RAG)** pipeline.  
It combines **document retrieval** with **LLM reasoning** and adds a **self-reflection loop** where the model evaluates its own answers for completeness and correctness.

---

## How It Works
1. **Retriever Node**: Fetches relevant chunks from the Xumo Stream Box manual using Hugging Face embeddings + FAISS.  
2. **Responder Node**: Groq LLM generates an initial answer based on the retrieved context.  
3. **Reflector Node**: The LLM reflects on its own answer, deciding if it fully addresses the question.  
   - If reflection = **YES**, the pipeline ends.  
   - If reflection = **NO**, the pipeline loops back to retrieval for another attempt.  
4. **Finalizer Node**: Produces the final answer, reflection log, and number of attempts.  

This iterative loop ensures the agent doesn’t stop at the first draft but **self-corrects** until a satisfactory answer is reached or a maximum attempt threshold is hit.

---

## Where It Works
- **Technical Manuals**: Devices like the Xumo Stream Box, where specifications and safety notes are critical.  
- **Enterprise Knowledge Bases**: Internal documentation where accuracy matters.  
- **Education & Training**: Teaching students how RAG pipelines can self-reflect to improve reliability.  
- **Customer Support Assistants**: Bots that need to check their own answers before responding to users.

---

## Example Query
**Asked:**  
*Is the Xumo Stream Box a 4K box?*

**RAG Reflection Output:**

- **Final Answer:**  
  The manual does not explicitly mention the resolution of the box. However, there is no mention of a limitation to 1080p or 720p resolutions, and considering the video connection type is HDMI (High-Definition Multimedia Interface), it is likely that the box supports 4K resolution, but the manual does not confirm this.

- **Reflection Log:**  
  Reflection: NO  
  Explanation: The answer does not directly answer the question "is it a 4K box?" but rather provides a possible inference. A more accurate answer would be:  
  *"The manual does not confirm if it is a 4K box"* or *"It is unknown if it is a 4K box."*

- **Total Attempts:** 2

---

## Takeaway
The **Reflection RAG pipeline** doesn’t just generate answers — it **evaluates and revises them**.  
This makes it especially useful when working with **manuals, technical documentation, or knowledge bases**, where precision and clarity are essential.



In [12]:
# ---------------------------------
# 0. Setup & Imports
# ---------------------------------
import os
from typing import List
from pydantic import BaseModel
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END

In [13]:
# Load environment variables
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# ---------------------------------
# 1. Prepare Knowledge Base
# ---------------------------------
manual = TextLoader(
    "C:/Users/admin/Desktop/New_GenAI/GenAI/LangGraph/Autonomus RAG/xumo_manual_rag.txt",
    encoding="utf-8"
).load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
segments = splitter.split_documents(manual)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
store = FAISS.from_documents(segments, embeddings)
retriever = store.as_retriever()

In [14]:
# ---------------------------------
# 2. Initialize Groq LLM
# ---------------------------------
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY")
)

In [15]:
# ---------------------------------
# 3. State Definition
# ---------------------------------
class ManualReflectState(BaseModel):
    question: str
    retrieved_docs: List[Document] = []
    answer: str = ""
    reflection: str = ""
    revised: bool = False
    attempts: int = 0

In [16]:
# ---------------------------------
# 4. Nodes
# ---------------------------------

# a. Retrieve
def fetch_docs(state: ManualReflectState) -> ManualReflectState:
    docs = retriever.invoke(state.question)
    return state.model_copy(update={"retrieved_docs": docs})

# b. Generate Answer
def draft_answer(state: ManualReflectState) -> ManualReflectState:
    context = "\n\n".join([doc.page_content for doc in state.retrieved_docs])
    prompt = f"""
Use the following manual excerpts to answer the question.

Context:
{context}

Question:
{state.question}
"""
    answer = llm.invoke(prompt).content.strip()
    return state.model_copy(update={"answer": answer, "attempts": state.attempts + 1})

# c. Self-Reflect
def review_answer(state: ManualReflectState) -> ManualReflectState:
    prompt = f"""
Reflect on the following answer. 
Say YES if it fully addresses the question, or NO with an explanation.

Question: {state.question}

Answer: {state.answer}

Respond like:
Reflection: YES or NO
Explanation: ...
"""
    result = llm.invoke(prompt).content
    is_ok = "reflection: yes" in result.lower()
    return state.model_copy(update={"reflection": result, "revised": not is_ok})

# d. Finalizer
def finalize(state: ManualReflectState) -> ManualReflectState:
    return state

In [17]:
# ---------------------------------
# 5. LangGraph DAG
# ---------------------------------
builder = StateGraph(ManualReflectState)

builder.add_node("retriever", fetch_docs)
builder.add_node("responder", draft_answer)
builder.add_node("reflector", review_answer)
builder.add_node("done", finalize)

builder.set_entry_point("retriever")

builder.add_edge("retriever", "responder")
builder.add_edge("responder", "reflector")
builder.add_conditional_edges(
    "reflector",
    lambda s: "done" if not s.revised or s.attempts >= 2 else "retriever"
)

builder.add_edge("done", END)
graph = builder.compile()

In [18]:
# ---------------------------------
# 6. Run the Agent
# ---------------------------------
if __name__ == "__main__":
    user_query = "Does the Xumo Stream Box support 4K resolution?"
    init_state = ManualReflectState(question=user_query)
    result = graph.invoke(init_state)

    print("\nFinal Answer:\n", result["answer"])
    print("\nReflection Log:\n", result["reflection"])
    print("Total Attempts:", result["attempts"])



Final Answer:
 Unfortunately, there is no information provided in the given manual excerpts about the Xumo Stream Box's support for 4K resolution.

Reflection Log:
 Reflection: NO
Explanation: The answer states that there is no information provided about the Xumo Stream Box's support for 4K resolution, but it does not confirm whether or not the device supports 4K resolution. It does not provide a clear positive or negative answer to the question.
Total Attempts: 2


In [19]:
# ---------------------------------
# 6. Gradio Interface
# ---------------------------------
def rag_reflection_pipeline(user_query: str):
    init_state = ManualReflectState(question=user_query)
    result = graph.invoke(init_state)
    return (
        f"Final Answer:\n{result['answer']}\n\n"
        f"Reflection Log:\n{result['reflection']}\n\n"
        f"Total Attempts: {result['attempts']}"
    )

demo = gr.Interface(
    fn=rag_reflection_pipeline,
    inputs=gr.Textbox(label="Ask about Xumo Stream Box Manual"),
    outputs=gr.Textbox(label="RAG Reflection Output"),
    title="Xumo Manual RAG Reflection Assistant"
)

if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
